# Fase 3.9 — notebook final de validação e fechamento

Este notebook **reúne dois notebooks anteriores**: `rodar_fase3_9_tgif_colab.ipynb` (preparação do runtime, cópia local e leitura dos resultados) e `validar_perguntas_fase3_9_colab.ipynb` (V1, V2 e V3). Os originais continuam preservados.

O preparo TGIF, o smoke test, as 12 épocas de treino, a calibração interna e a avaliação comparativa Mirage **já foram executados**. Este notebook não contém comandos para repetir treino, recalibrar ou inferir no Mirage. A etapa restante da 3.9 é validar as perguntas V1–V3, preservar os registros e conferir o fechamento documental. Isso não significa alcançar 85% de acurácia externa.

Execute as células de código **1–8**, depois **9–12** se for concluir V2/V3. A célula 9 copia dados internos para `/content` e pode demorar. V2 pode percorrer arquivos TGIF compactados; V3 calcula hashes de 12.475 imagens e treina apenas um baseline estatístico de tamanho em dados internos. Não usa imagens Mirage para treino ou escolha de parâmetros. **GPU não é necessária.**

In [ ]:
# 1. Dependências da validação; GPU não é necessária.
import importlib.util, subprocess, sys
required_packages = {'cryptography': 'cryptography', 'numpy': 'numpy',
                     'PIL': 'Pillow', 'sklearn': 'scikit-learn'}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
print('Dependências disponíveis. Este notebook não inicia treino nem inferência externa.', flush=True)

In [ ]:
# 2. Montagem do Drive e definição da raiz do projeto.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_9', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

In [ ]:
# 3. Desbloqueio da credencial GitHub cifrada; sem exibir o token.
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot imutável da branch privada em /content; somente leitura no GitHub.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase39-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('Notebooks/fase_3/fase3_9_final_colab.ipynb', 'experimentos/espelhar_codigo_drive.py', 'experimentos/fase3_8_aigi_piloto.py', 'experimentos/fase3_8_aigi_dados.py', 'experimentos/fase3_9_tgif_piloto.py', 'experimentos/fase3_9_tgif_dados.py', 'experimentos/fase3_9_tgif_finetune.py', 'experimentos/avaliar_fase3_9_mirage.py', 'experimentos/fase3_5_dados.py', 'experimentos/fase3_7_dados.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc39_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.9 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Preparar caminhos e indicador de progresso; nenhuma chamada de treino.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_9_tgif_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_9_tgif_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

## Espelho e estado já concluído

A célula 6 copia este próprio notebook, os demais notebooks e scripts versionados para as respectivas pastas do Drive, preservando versões divergentes em `codigo_historico/`. A célula 7 só lê os artefatos da 3.9.

In [ ]:
# 6. Copiar este notebook e os demais .ipynb/.py para as mesmas pastas no Drive.
# O script guarda em codigo_historico/ qualquer versão diferente já existente.
import hashlib
mirror_script = CODE_ROOT / 'experimentos/espelhar_codigo_drive.py'
notebook_relative = Path('Notebooks/fase_3/fase3_9_final_colab.ipynb')
notebook_source = CODE_ROOT / notebook_relative
if not notebook_source.is_file() or not mirror_script.is_file():
    raise FileNotFoundError('Publique este notebook e o script de espelho na branch antes de sincronizar')
run_live('espelho do código no Drive',
         [sys.executable, '-u', str(mirror_script), '--source', str(CODE_ROOT),
          '--drive-root', str(ROOT), '--commit', commit])
notebook_drive = ROOT / notebook_relative
def file_sha256(path):
    value = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()
if not notebook_drive.is_file() or file_sha256(notebook_drive) != file_sha256(notebook_source):
    raise IOError('A cópia deste notebook no Drive não passou na verificação de hash')
print('Notebook final no Drive:', notebook_drive)
print('Manifesto de código:', ROOT / 'codigo_sync' / f'{commit}.json')

In [ ]:
# 7. Conferir artefatos já concluídos, sem inferência ou alteração dos resultados.
run_dir = ROOT / 'modelos/fase3_9_tgif_sd2fr_v1'
mirage_dir = ROOT / 'resultados/benchmark_externo_mirage_2000_v1'
prior = {
    'treino': run_dir / 'training_complete.json',
    'checkpoint': run_dir / 'phase3_final.pt',
    'metricas_internas': run_dir / 'internal_metrics.json',
    'divisoes': run_dir / 'split_manifest.json',
    'bloqueio_mirage': mirage_dir / 'locks/fase3_9_tgif_sd2fr_v1.json',
    'metricas_mirage': mirage_dir / 'fase3_9_tgif_sd2fr_v1/metrics.json',
}
for label, path in prior.items():
    print(label, 'presente' if path.is_file() else 'AUSENTE', '|', path)
missing = [label for label, path in prior.items() if not path.is_file()]
if missing:
    raise FileNotFoundError('Artefatos esperados ausentes: ' + ', '.join(missing))
internal = json.loads(prior['metricas_internas'].read_text(encoding='utf-8'))
external = json.loads(prior['metricas_mirage'].read_text(encoding='utf-8'))
print('Seleção interna AUC:', internal.get('selection_auc'),
      '| melhor época:', internal.get('best_epoch'))
print('Mirage comparativo acurácia:', external.get('accuracy'),
      '| AUC:', external.get('auc'),
      '| recall sintético:', external.get('recall_fake'),
      '| especificidade real:', external.get('specificity_real'))
print('Mirage já observado: estes valores não são usados para ajustar a fase.')

## Pergunta V1 — código, configuração e resultados registrados

Esta célula lê arquivos existentes. Seus textos são diagnóstico; não selecionam novos hiperparâmetros pelo Mirage.

In [ ]:
# 8. V1 — configuração, código e resultados já gravados; somente leitura.
import json, re
from pathlib import Path
if any(n not in globals() for n in ('ROOT', 'CODE_ROOT')):
    raise RuntimeError('Execute antes as células 1 a 5 do notebook 3.9 (definem ROOT e CODE_ROOT).')
RUN = ROOT / 'modelos/fase3_9_tgif_sd2fr_v1'
OUT = ROOT / 'resultados/validacao_perguntas_3_9_v1'
OUT.mkdir(parents=True, exist_ok=True)

def cortar(texto, n):
    return texto if len(texto) <= n else texto[:n] + ' ...[cortado]'

print('== experiment_config.json ==')
cfg_path = RUN / 'experiment_config.json'
if cfg_path.is_file():
    print(cortar(json.dumps(json.loads(cfg_path.read_text(encoding='utf-8')), indent=1, ensure_ascii=False), 6000))
else:
    print('AUSENTE:', cfg_path)

PADRAO = re.compile(r'label_smooth|smoothing|CrossEntropy|BCE|[Ff]ocal|[Rr]esize|RandomResizedCrop|RandomCrop|'
                    r'CenterCrop|interpolation|[Bb]lur|JPEG|jpeg|quality|ColorJitter|Flip|augment|IMG_SIZE|'
                    r'image_size|input_size|WeightedRandom|sampler|\bsqrt\b|thumbnail|\.save\(|convert\(')

def grep_linhas(nome, linhas, maximo=70):
    achadas = [(i, l.strip()) for i, l in enumerate(linhas, 1) if PADRAO.search(l)]
    print(f'\n== {nome}: {len(achadas)} linhas com termos de interesse (mostrando {min(maximo, len(achadas))}) ==')
    for i, l in achadas[:maximo]:
        print(f'{i:5d}: {cortar(l, 170)}')

EXP = CODE_ROOT / 'experimentos'
for nome in ('fase3_9_tgif_finetune.py', 'fase3_9_tgif_dados.py', 'fase3_8_aigi_dados.py',
             'fase3_7_dados.py', 'fase3_5_dados.py'):
    caminho = EXP / nome
    if not caminho.is_file():
        print(f'\n== {nome}: AUSENTE ==')
        continue
    texto = caminho.read_text(encoding='utf-8').splitlines()
    if nome == 'fase3_9_tgif_finetune.py':
        print('\n== imports locais em', nome, '==')
        for i, l in enumerate(texto, 1):
            if re.match(r'\s*(from|import)\s+\w*(fase3|avaliar|benchmark|experimentos)', l):
                print(f'{i:5d}: {l.strip()}')
    grep_linhas(nome, texto)

fonte = CODE_ROOT / 'Notebooks/fase_3/fase3.3_compressao.ipynb'
if fonte.is_file():
    nb = json.loads(fonte.read_text(encoding='utf-8'))
    linhas_nb = []
    for cel in nb.get('cells', []):
        if cel.get('cell_type') == 'code':
            src = cel.get('source', [])
            linhas_nb += (src.splitlines() if isinstance(src, str) else ''.join(src).splitlines())
    grep_linhas('fase3.3_compressao.ipynb (apenas células de código)', linhas_nb)

print('\n== Resultados já gravados (metrics.json / summary.json; somente leitura) ==')
achados = [p for p in sorted((ROOT / 'resultados').rglob('*.json'))
           if p.name in ('metrics.json', 'summary.json') and p.stat().st_size < 200_000][:30]
if not achados:
    print('Nenhum metrics.json/summary.json em', ROOT / 'resultados')
for p in achados:
    print('\n#', p.relative_to(ROOT))
    print(cortar(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=1, ensure_ascii=False), 1800))


## Dados locais para V2 e V3

Execute a célula 9 uma vez por sessão. Ela copia dados internos congelados do Drive para `/content`, com indicadores de progresso, sem baixar novas fontes nem treinar.

In [ ]:
# 9. Copiar dados internos congelados para /content; executar antes de V2/V3.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
CF_DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
AIGI_DATA = CODE_ROOT / 'experimentos/fase3_8_aigi_dados.py'
for label, script in [('CF 3.5', OLD_DATA), ('CF 3.7', CF_DATA), ('AIGI 3.8', AIGI_DATA), ('TGIF 3.9', DATA)]:
    run_live('copia local '+label, [sys.executable, '-u', str(script), 'stage', '--root', str(ROOT)])


## Perguntas V2 e V3

V2 examina pares TGIF, formatos e qualidade JPEG estimada. V3 verifica hashes, candidatos visuais e um baseline estatístico que vê dimensões e bytes; a calibração aparece somente como diagnóstico, sem alterar o detector. Cada execução grava um **novo** JSON com data e hora UTC e usa abertura exclusiva, preservando logs anteriores.

In [ ]:
# 10. V2 — pares TGIF, formatos e qualidade; relatório com data/hora UTC.
import json, re, time, tarfile
from collections import Counter
from pathlib import Path
import numpy as np
from PIL import Image
if any(n not in globals() for n in ('ROOT', 'CODE_ROOT')):
    raise RuntimeError('Execute antes as células 1 a 5 do notebook 3.9 (definem ROOT e CODE_ROOT).')
RUN = ROOT / 'modelos/fase3_9_tgif_sd2fr_v1'
OUT = ROOT / 'resultados/validacao_perguntas_3_9_v1'
OUT.mkdir(parents=True, exist_ok=True)
resumo = {}

def origem(p):
    p = str(p)
    for chave, nome in (('aigi_pares_256_v1', 'aigi'), ('tgif_sd2fr_768_v1', 'tgif'),
                        ('fase3_5_novos_v1', 'cfsmall'), ('fase3_7_cfsmall_expansao_v1', 'cfsmall'),
                        ('dados_fase3_4', 'openfake')):
        if chave in p:
            return nome
    return 'outro'

def achatar(obj, rotulo=''):
    if isinstance(obj, list):
        for x in obj:
            if isinstance(x, dict) and 'path' in x:
                yield rotulo, x
            else:
                yield from achatar(x, rotulo)
    elif isinstance(obj, dict):
        for k, v in obj.items():
            yield from achatar(v, f'{rotulo}/{k}' if rotulo else str(k))

def pct(valores, qs=(10, 50, 90)):
    return [round(float(np.percentile(valores, q)), 3) for q in qs] if len(valores) else []

# ---- B1: conteúdo dos .tar.gz do TGIF (lista nomes; não extrai nada) ----
print('== B1. Arquivos TGIF baixados (só lista nomes; limite de 150 s por arquivo) ==')
tars = sorted((ROOT / 'dados_fase3_9').rglob('tgif*.tar.gz')) + sorted(Path('/content').glob('tgif*.tar.gz'))
if not tars:
    print('Nenhum tgif*.tar.gz em dados_fase3_9 nem em /content.')
META = re.compile(r'mask|msk|annot|label|gt_|readme|\.md$|\.txt$|\.json$|\.csv$|\.parquet$', re.I)
resumo['tgif_tar'] = {}
for caminho in tars:
    inicio = ultimo_aviso = time.monotonic()
    topo, ext, amostra, meta, n, parcial = Counter(), Counter(), [], [], 0, False
    try:
        with tarfile.open(caminho, 'r|gz') as tf:
            for m in tf:
                n += 1
                if time.monotonic() - ultimo_aviso >= 30:
                    print(f'{caminho.name}: {n} membros listados em {(time.monotonic()-inicio)/60:.1f} min', flush=True)
                    ultimo_aviso = time.monotonic()
                partes = m.name.strip('./').split('/')
                topo['/'.join(partes[:2]) if len(partes) > 2 else partes[0]] += 1
                ext[Path(m.name).suffix.lower()] += 1
                if m.isfile() and len(amostra) < 6:
                    amostra.append(m.name)
                if META.search(m.name) and len(meta) < 12:
                    meta.append(m.name)
                if n >= 300000 or time.monotonic() - inicio > 150:
                    parcial = True
                    break
    except Exception as erro:
        print(caminho.name, 'erro ao listar:', type(erro).__name__, erro)
        continue
    print(f'\n# {caminho} | {caminho.stat().st_size / 1e9:.2f} GB | membros lidos: {n}'
          f'{" (LISTAGEM PARCIAL)" if parcial else ""}')
    print('  diretórios (2 níveis, top 10):', topo.most_common(10))
    print('  extensões:', ext.most_common(8))
    print('  exemplos:', amostra)
    print('  nomes tipo máscara/anotação/metadado:', meta if meta else 'nenhum encontrado na parte lida')
    resumo['tgif_tar'][caminho.name] = dict(membros=n, parcial=parcial, ext=dict(ext.most_common(8)), meta=meta)

# ---- B2: pares do manifesto TGIF local ----
print('\n== B2. Manifesto TGIF local: estrutura e pareamento ==')
cands = [Path('/content/tgif_sd2fr_768_v1/manifest.json'), ROOT / 'dados_fase3_9/tgif_sd2fr_768_v1/manifest.json']
mpath = next((p for p in cands if p.is_file()), None)
if mpath is None:
    print('manifest.json do TGIF não encontrado (execute a célula 7 do notebook 3.9).')
else:
    man = json.loads(mpath.read_text(encoding='utf-8'))
    recs = list(achatar(man))
    print('Manifesto:', mpath, '| tipo:', type(man).__name__,
          '| chaves de topo:', list(man)[:12] if isinstance(man, dict) else '-')
    print('Registros com "path":', len(recs))
    if recs:
        print('Primeiro registro:', json.dumps(recs[0][1], ensure_ascii=False)[:500])
        print('Campos presentes:', Counter(k for _, r in recs for k in r).most_common(15))
        print('Rótulos de partição x classe:', Counter((s, r.get('label')) for s, r in recs).most_common(12))
    CHAVES = ('pair_id', 'par_id', 'pair', 'par', 'stem', 'source_id', 'orig_id', 'source', 'id', 'key')
    chave = next((k for k in CHAVES if recs and all(k in r for _, r in recs)), None)
    grupos = {}
    if chave:
        print('Pareamento pela chave explícita:', chave)
        for s, r in recs:
            grupos.setdefault((s.split('/')[0], r[chave]), []).append(r)
    else:
        print('Nenhuma chave explícita de par encontrada; tentando nome de arquivo idêntico entre classes.')
        for s, r in recs:
            grupos.setdefault((s.split('/')[0], Path(r['path']).name), []).append(r)
    pares = [v for v in grupos.values() if len(v) == 2 and {x['label'] for x in v} == {0, 1}]
    print('Grupos formados:', len(grupos), '| pares real+fake (1+1):', len(pares))
    resumo['tgif_pares_identificados'] = len(pares)
    if not pares:
        print('PAREAMENTO NÃO IDENTIFICÁVEL pelo manifesto. Cole aqui o "Primeiro registro" acima.')
    else:
        medidas = []
        passo = max(1, len(pares) // 128)
        for v in pares[::passo][:128]:
            real = next(x for x in v if x['label'] == 0)
            fake = next(x for x in v if x['label'] == 1)
            try:
                with Image.open(real['path']) as a, Image.open(fake['path']) as b:
                    sa, sb = a.size, b.size
                    mesmo = sa == sb
                    ga, gb = a.convert('L'), b.convert('L')
                    if not mesmo:
                        ga, gb = ga.resize((128, 128), Image.Resampling.BILINEAR), gb.resize((128, 128), Image.Resampling.BILINEAR)
                    d = np.abs(np.asarray(ga, dtype=np.float32) - np.asarray(gb, dtype=np.float32))
                medidas.append(dict(mesmo=mesmo, aspecto=abs(sa[0] / sa[1] - sb[0] / sb[1]) < 0.01,
                                    mae=float(d.mean()), frac16=float((d > 16).mean())))
            except Exception as erro:
                print('par ignorado:', type(erro).__name__, erro)
        if medidas:
            n = len(medidas)
            print(f'Pares medidos: {n} | mesmo tamanho: {sum(m["mesmo"] for m in medidas)}/{n}'
                  f' | mesma proporção: {sum(m["aspecto"] for m in medidas)}/{n}')
            print('MAE em cinza (p10/p50/p90):', pct([m['mae'] for m in medidas]),
                  '| fração de pixels com |diff|>16 (p10/p50/p90):', pct([m['frac16'] for m in medidas]))
            print('Leitura: fração baixa = só parte da imagem mudou; alta = imagem toda diferente. '
                  'Comparação feita nos arquivos locais (já copiados/possivelmente reprocessados).')
            resumo['tgif_pares'] = dict(n=n, mesmo_tamanho=sum(m['mesmo'] for m in medidas),
                                         mae=pct([m['mae'] for m in medidas]), frac16=pct([m['frac16'] for m in medidas]))

# ---- B3: formato e qualidade JPEG por origem x classe (treino, amostra) ----
print('\n== B3. Formato e qualidade JPEG estimada (amostra do treino) ==')
STD = [16,11,10,16,24,40,51,61,12,12,14,19,26,58,60,55,14,13,16,24,40,57,69,56,14,17,22,29,51,87,80,62,
       18,22,37,56,68,109,103,77,24,35,55,64,81,104,113,92,49,64,78,87,103,121,120,101,72,92,95,98,112,100,103,99]
ZZ = [0,1,8,16,9,2,3,10,17,24,32,25,18,11,4,5,12,19,26,33,40,48,41,34,27,20,13,6,7,14,21,28,
      35,42,49,56,57,50,43,36,29,22,15,23,30,37,44,51,58,59,52,45,38,31,39,46,53,60,61,54,47,55,62,63]
STD_ZZ = [STD[i] for i in ZZ]

def qualidade(im):
    q = getattr(im, 'quantization', None)
    if not q or 0 not in q:
        return None, None
    t, melhor = list(q[0]), None
    for Q in range(1, 101):
        s = 5000 / Q if Q < 50 else 200 - 2 * Q
        for base in (STD, STD_ZZ):
            ref = [min(255, max(1, int((v * s + 50) // 100))) for v in base]
            erro = sum(abs(a - b) for a, b in zip(ref, t))
            if melhor is None or erro < melhor[0]:
                melhor = (erro, Q)
    return melhor[1], melhor[0]

sm = json.loads((RUN / 'split_manifest.json').read_text(encoding='utf-8'))
por_celula = {}
for r in sm['train']:
    por_celula.setdefault((origem(r['path']), int(r['label'])), []).append(r)
resumo['formato'] = {}
for (org, lab), rs in sorted(por_celula.items()):
    rs = sorted(rs, key=lambda r: r['path'])
    rs = rs[::max(1, len(rs) // 200)][:200]
    fmts, quals, erros, curtos, bpp, falta = Counter(), [], [], [], [], 0
    for r in rs:
        p = Path(r['path'])
        if not p.is_file():
            falta += 1
            continue
        with Image.open(p) as im:
            fmts[im.format] += 1
            w, h = im.size
            curtos.append(min(w, h))
            bpp.append(p.stat().st_size / (w * h))
            if im.format == 'JPEG':
                q, e = qualidade(im)
                if q is not None:
                    quals.append(q)
                    erros.append(e)
    print(f'{org:8s} classe={lab} n={len(rs) - falta:3d} (ausentes={falta}) formatos={dict(fmts)} '
          f'lado_curto p10/50/90={pct(curtos, (10, 50, 90))} bytes/pixel p50={pct(bpp, (50,))} '
          f'qualidade_JPEG_estimada p10/50/90={pct(quals)} | tabela distante do padrão (erro>40): {sum(e > 40 for e in erros)}')
    resumo['formato'][f'{org}_{lab}'] = dict(n=len(rs) - falta, formatos=dict(fmts), q=pct(quals), curto=pct(curtos))

print('\n== B4. Campos de origem nos manifestos locais (orig_format etc.) ==')
for nome in ('fase3_5_novos_v1', 'fase3_7_cfsmall_expansao_v1', 'aigi_pares_256_v1', 'tgif_sd2fr_768_v1'):
    p = Path('/content') / nome / 'manifest.json'
    if not p.is_file():
        print(nome, ': manifest local ausente (execute a célula 7)')
        continue
    rs = [r for _, r in achatar(json.loads(p.read_text(encoding='utf-8')))]
    campos = Counter(k for r in rs for k in r)
    print(nome, '| registros:', len(rs), '| campos:', list(campos)[:14])
    if any('orig_format' in r for r in rs):
        print('   orig_format:', dict(Counter(str(r.get('orig_format')).upper() for r in rs)))

from datetime import datetime, timezone
V2_REPORT = OUT / ('v2_tgif_e_formato_' +
                   datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f_UTC') + '.json')
with V2_REPORT.open('x', encoding='utf-8') as stream:
    json.dump(resumo, stream, indent=1, ensure_ascii=False)
print('\nResumo gravado sem substituir relatórios anteriores:', V2_REPORT)


In [ ]:
# 11. V3 — duplicatas e baseline de tamanho; relatório com data/hora UTC.
# Abre as imagens do teste reservado apenas para hash/tamanho (como o smoke já faz); nenhuma inferência de modelo.
import csv, io, json, time, hashlib
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
if any(n not in globals() for n in ('ROOT', 'CODE_ROOT')):
    raise RuntimeError('Execute antes as células 1 a 5 do notebook 3.9 (definem ROOT e CODE_ROOT).')
RUN = ROOT / 'modelos/fase3_9_tgif_sd2fr_v1'
OUT = ROOT / 'resultados/validacao_perguntas_3_9_v1'
OUT.mkdir(parents=True, exist_ok=True)
resumo = {}
PARTES = ('train', 'validation_selection', 'validation_calibration', 'test_reserved')

def origem(p):
    p = str(p)
    for chave, nome in (('aigi_pares_256_v1', 'aigi'), ('tgif_sd2fr_768_v1', 'tgif'),
                        ('fase3_5_novos_v1', 'cfsmall'), ('fase3_7_cfsmall_expansao_v1', 'cfsmall'),
                        ('dados_fase3_4', 'openfake')):
        if chave in p:
            return nome
    return 'outro'

sm = json.loads((RUN / 'split_manifest.json').read_text(encoding='utf-8'))
print('Partições:', {k: len(sm[k]) for k in PARTES})
todos = [(parte, r) for parte in PARTES for r in sm[parte]]
linhas, faltam = [], Counter()
inicio = ultimo = time.monotonic()
for i, (parte, r) in enumerate(todos, 1):
    p = Path(r['path'])
    if not p.is_file():
        faltam[parte] += 1
        continue
    bruto = p.read_bytes()
    with Image.open(io.BytesIO(bruto)) as im:
        w, h = im.size
        pix = list(im.convert('L').resize((9, 8), Image.Resampling.BILINEAR).getdata())
    dh = sum((pix[y * 9 + x] > pix[y * 9 + x + 1]) << (y * 8 + x) for y in range(8) for x in range(8))
    linhas.append(dict(parte=parte, label=int(r['label']), group=r['group'], path=r['path'], origem=origem(r['path']),
                       sha=hashlib.sha256(bruto).hexdigest(), dh=dh, w=w, h=h, bytes=len(bruto)))
    if time.monotonic() - ultimo >= 30:
        print(f'Hashes: {i}/{len(todos)} em {(time.monotonic() - inicio) / 60:.1f} min', flush=True)
        ultimo = time.monotonic()
print('Arquivos ausentes por partição:', dict(faltam) or 'nenhum')
if faltam:
    print('ATENÇÃO: há arquivos ausentes; confirme que a célula 7 do notebook 3.9 foi executada nesta sessão.')

# ---- C1: duplicatas exatas ----
print('\n== C1. Duplicatas exatas (SHA-256) ==')
por_sha = defaultdict(list)
for l in linhas:
    por_sha[l['sha']].append(l)
dups = [v for v in por_sha.values() if len(v) > 1]
cruz = [v for v in dups if len({x['parte'] for x in v}) > 1]
conf = [v for v in dups if len({x['label'] for x in v}) > 1]
print(f'Grupos duplicados: {len(dups)} | cópias extras: {sum(len(v) - 1 for v in dups)} | '
      f'entre partições diferentes: {len(cruz)} | com rótulos conflitantes: {len(conf)}')
print('Duplicatas dentro de uma mesma partição:', dict(Counter(v[0]['parte'] for v in dups if len({x['parte'] for x in v}) == 1)))
for v in cruz[:15]:
    print('  cruzada:', [(x['parte'], x['label'], x['group'], Path(x['path']).name) for x in v])
resumo['duplicatas_exatas'] = dict(grupos=len(dups), cruzadas=len(cruz), conflitantes=len(conf))

# ---- C2: candidatos visuais (dHash <= 4), apenas candidatos ----
print('\n== C2. Candidatos visuais (dHash<=4; exigem revisão; não confirmam duplicação) ==')
_T = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def popcount(x):
    try:
        return np.bitwise_count(x)
    except AttributeError:
        return _T[np.ascontiguousarray(x).view(np.uint8).reshape(*x.shape, 8)].sum(-1)

idx = {p: [l for l in linhas if l['parte'] == p] for p in PARTES}
H = {p: np.array([l['dh'] for l in idx[p]], dtype=np.uint64) for p in PARTES}
combos = (('train', 'validation_selection'), ('train', 'validation_calibration'), ('train', 'test_reserved'),
          ('validation_selection', 'validation_calibration'), ('validation_selection', 'test_reserved'),
          ('validation_calibration', 'test_reserved'))
achados, contagem, perto = [], Counter(), Counter()
for a, b in combos:
    if not len(H[a]) or not len(H[b]):
        continue
    for c0 in range(0, len(H[a]), 256):
        d = popcount(H[a][c0:c0 + 256, None] ^ H[b][None, :])
        for ia, ib in zip(*np.nonzero(d <= 4)):
            la, lb = idx[a][c0 + ia], idx[b][ib]
            if la['sha'] == lb['sha']:
                continue
            dist = int(d[ia, ib])
            achados.append((dist, a, b, la, lb))
            contagem[(a, b, la['label'], lb['label'])] += 1
            perto[(a, b)] += dist <= 2
for (a, b, ya, yb), n in sorted(contagem.items()):
    print(f'  {a} x {b} | classes ({ya},{yb}): {n}')
print('  Com dHash<=2:', {f'{a} x {b}': int(n) for (a, b), n in perto.items()})
print('  Total de candidatos:', len(achados))
resumo['candidatos_dhash'] = {f'{a}|{b}|{ya}|{yb}': n for (a, b, ya, yb), n in contagem.items()}

def rgb_pequeno(caminho):
    with Image.open(caminho) as im:
        return np.asarray(im.convert('RGB').resize((96, 96), Image.Resampling.BICUBIC), dtype=np.float32)
print('  Revisão numérica dos 30 candidatos mais próximos (MAE RGB baixo sugere semelhança, não prova):')
for dist, a, b, la, lb in sorted(achados, key=lambda t: t[0])[:30]:
    mae = float(np.abs(rgb_pequeno(la['path']) - rgb_pequeno(lb['path'])).mean())
    print(f'   dHash={dist} MAE={mae:6.2f} | {a[:5]}:{la["label"]}:{la["group"]}:{Path(la["path"]).name}'
          f' | {b[:5]}:{lb["label"]}:{lb["group"]}:{Path(lb["path"]).name}')

# ---- C3: baseline "só tamanho/bytes" ----
print('\n== C3. Baseline que vê só dimensões e bytes do arquivo (treino -> seleção/calibração) ==')
def feats(L, modo):
    w, h, b = (np.array([l[k] for l in L], dtype=float) for k in ('w', 'h', 'bytes'))
    s, m = np.minimum(w, h), np.maximum(w, h)
    if modo == 'curto':
        return np.c_[np.log(s)]
    return np.c_[np.log(s), np.log(m), np.log(b), np.log(b / (w * h))]

def auc_baseline(treino, aval, modo):
    y, ya = [l['label'] for l in treino], [l['label'] for l in aval]
    if len(set(y)) < 2 or len(set(ya)) < 2:
        return None
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(feats(treino, modo), y)
    return round(float(roc_auc_score(ya, clf.predict_proba(feats(aval, modo))[:, 1])), 3)

resumo['baseline_tamanho'] = {}
for alvo in ('validation_selection', 'validation_calibration'):
    print(f'\nAvaliação em {alvo} (AUC; 0,5 = sem sinal; "curto" = só lado curto; "tudo" = lado curto, longo, bytes, bytes/pixel)')
    for org in ['(todas)'] + sorted({l['origem'] for l in idx['train']}):
        tr = idx['train'] if org == '(todas)' else [l for l in idx['train'] if l['origem'] == org]
        av = idx[alvo] if org == '(todas)' else [l for l in idx[alvo] if l['origem'] == org]
        r = {m: auc_baseline(tr, av, m) for m in ('curto', 'tudo')}
        print(f'  {org:9s} treino n={len(tr):5d} classes={dict(Counter(l["label"] for l in tr))} | '
              f'avaliação n={len(av):4d} classes={dict(Counter(l["label"] for l in av))} | AUC {r}')
        resumo['baseline_tamanho'][f'{alvo}|{org}'] = r

# ---- C4: erros do modelo por faixa de tamanho (se a auditoria 3.9 já existir) ----
seg = ROOT / 'resultados/auditoria_interna_fase3_9_v1/segments_selection_3_9.csv'
print('\n== C4. Erros do modelo 3.9 por faixa de tamanho (lido de auditoria já gravada) ==')
if seg.is_file():
    with seg.open(newline='', encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            if row.get('dimension') == 'dataset_size_band' and row.get('value') and int(row['n']) >= 10:
                print('  ', {k: row[k] for k in ('value', 'label', 'n', 'errors', 'error_rate') if k in row})
else:
    print('Arquivo ausente:', seg, '(a auditoria interna 3.9 não foi executada nesta conta/pasta).')

resumo['arquivos_ausentes'] = dict(faltam)
from datetime import datetime, timezone
V3_REPORT = OUT / ('v3_duplicatas_e_baseline_' +
                   datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f_UTC') + '.json')
with V3_REPORT.open('x', encoding='utf-8') as stream:
    json.dump(resumo, stream, indent=1, ensure_ascii=False)
print('\nResumo gravado sem substituir relatórios anteriores:', V3_REPORT)


## Fechamento documental

A célula 12 confere os arquivos das fases interna e Mirage já concluídas, compara hashes e valores congelados e registra um novo resumo com data e hora. Ela não faz inferência nem muda pesos. Avisos no relatório exigem revisão; o Mirage já observado não demonstra a meta externa independente.

In [ ]:
# 12. Conferência final documental da Fase 3.9; não reavalia o Mirage.
import hashlib
from datetime import datetime, timezone
if 'V2_REPORT' not in globals() or 'V3_REPORT' not in globals():
    raise RuntimeError('Execute V2 e V3 nesta sessão antes da conferência final')
if not V2_REPORT.is_file() or not V3_REPORT.is_file():
    raise FileNotFoundError('Um dos relatórios V2/V3 desta sessão está ausente')
complete = json.loads(prior['treino'].read_text(encoding='utf-8'))
lock = json.loads(prior['bloqueio_mirage'].read_text(encoding='utf-8'))
internal = json.loads(prior['metricas_internas'].read_text(encoding='utf-8'))
external = json.loads(prior['metricas_mirage'].read_text(encoding='utf-8'))
v2 = json.loads(V2_REPORT.read_text(encoding='utf-8'))
v3 = json.loads(V3_REPORT.read_text(encoding='utf-8'))
if complete.get('status') != 'internal_training_complete':
    raise ValueError('O marcador de treino interno não confirma conclusão')
if internal.get('external_used') is not False or internal.get('openfake_test_used') is not False:
    raise ValueError('Separação interna/externa diferente da registrada')
if external.get('model') != 'fase3_9_tgif_sd2fr_v1' or external.get('n') != 2000:
    raise ValueError('Resultado Mirage não corresponde à Fase 3.9 e 2.000 imagens')
for key, internal_key in (('temperature_frozen', 'temperature'),
                          ('threshold_frozen', 'threshold')):
    if abs(float(external[key]) - float(internal[internal_key])) > 1e-8:
        raise ValueError(f'{key} difere da calibração interna')
print('Conferindo SHA-256 do checkpoint final (leitura de cerca de 40 MB)...', flush=True)
checkpoint_hash = file_sha256(prior['checkpoint'])
if checkpoint_hash != external.get('checkpoint_sha256') or checkpoint_hash != lock.get('checkpoint_sha256'):
    raise ValueError('Checkpoint diverge dos hashes congelados na avaliação')
if external.get('manifest_sha256') != lock.get('manifest_sha256'):
    raise ValueError('Manifesto Mirage diverge do bloqueio da avaliação')
duplicates = v3.get('duplicatas_exatas', {})
warnings = []
if v3.get('arquivos_ausentes'):
    warnings.append('Há imagens internas ausentes; confira a cópia local e o relatório V3.')
if int(duplicates.get('cruzadas', 0)) > 0 or int(duplicates.get('conflitantes', 0)) > 0:
    warnings.append('Há duplicatas entre divisões ou rótulos conflitantes; revisar antes de concluir.')
if not v2.get('tgif_pares_identificados'):
    warnings.append('V2 não identificou pares TGIF pelo manifesto; revisar a saída B2.')
if v3.get('candidatos_dhash'):
    warnings.append('Candidatos visuais dHash exigem revisão; similaridade não prova duplicata.')
report = {
    'experiment': 'fase3_9_tgif_sd2fr_v1',
    'status': 'revisao_tecnica_pendente' if warnings else 'validacoes_executadas',
    'code_commit': commit,
    'checkpoint_sha256': checkpoint_hash,
    'internal_selection_auc_descriptive': internal['selection_auc'],
    'mirage_accuracy_comparative': external['accuracy'],
    'mirage_auc_comparative': external['auc'],
    'mirage_already_observed': True,
    'blind_independent_final_test': False,
    'goal_note': 'A meta de 85% em teste externo independente ainda não foi demonstrada.',
    'v2_report': str(V2_REPORT),
    'v3_report': str(V3_REPORT),
    'warnings': warnings,
}
from datetime import datetime, timezone
closure_path = OUT / ('fechamento_fase3_9_' +
                      datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f_UTC') + '.json')
with closure_path.open('x', encoding='utf-8') as stream:
    json.dump(report, stream, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Registro de fechamento preservado:', closure_path)
if warnings:
    print('Relatórios salvos; revise os avisos técnicos antes de encerrar a auditoria da 3.9.')
else:
    print('Validações da 3.9 documentadas; a meta de 85% externo permanece em aberto.')